# Fraud Detection — Metrics Dashboard & Data Overview

This notebook provides:
1. **Dataset summary** — shape, class balance, nulls, cross-dataset alignment
2. **Pre-training data quality checks** — synthetic data integrity, feature distributions
3. **Post-training metrics tracking** — load CV results from all 3 models and visualize
4. **Ensemble analysis** — combined predictions, threshold tuning, final metrics

**Run this locally** (not on Colab) — all datasets and prediction CSVs are expected in the project directory.

In [ ]:
# Dependencies
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    f1_score, accuracy_score, roc_auc_score,
)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.bbox'] = 'tight'

DATA_DIR = Path('processed_data')
print('Ready')

## 1. Dataset Overview

In [ ]:
# Load all datasets
text_df    = pd.read_csv(DATA_DIR / 'text_dataset.csv')
image_df   = pd.read_csv(DATA_DIR / 'image_dataset.csv')
meta_df    = pd.read_csv(DATA_DIR / 'metadata_dataset.csv')
synth_text = pd.read_csv(DATA_DIR / 'synthetic_text_dataset.csv')
synth_meta = pd.read_csv(DATA_DIR / 'synthetic_metadata_dataset.csv')

datasets = {
    'Text (real)':       text_df,
    'Image (real)':      image_df,
    'Metadata (real)':   meta_df,
    'Text (synthetic)':  synth_text,
    'Meta (synthetic)':  synth_meta,
}

# Summary table
rows = []
for name, df in datasets.items():
    n = len(df)
    fraud = df['fraud_label'].sum()
    legit = n - fraud
    pct = 100 * fraud / n
    nulls = df.isnull().sum().sum()
    rows.append({'Dataset': name, 'Rows': n, 'Cols': df.shape[1],
                 'Legit': legit, 'Fraud': fraud, 'Fraud %': f'{pct:.1f}%',
                 'Total Nulls': nulls})

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

In [ ]:
# Class balance visualization
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (name, df) in zip(axes, [('Text', text_df), ('Image', image_df), ('Metadata', meta_df)]):
    vc = df['fraud_label'].value_counts().sort_index()
    colors = ['#2ecc71', '#e74c3c']
    bars = ax.bar(['Legit', 'Fraud'], vc.values, color=colors, edgecolor='white', linewidth=1.5)
    ax.set_title(f'{name} Dataset', fontweight='bold')
    ax.set_ylabel('Count')
    for bar, val in zip(bars, vc.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                str(val), ha='center', fontweight='bold')

plt.suptitle('Class Distribution — Real Datasets (before synthetic augmentation)', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Cross-dataset alignment
text_pids = set(text_df['product_id'])
meta_pids = set(meta_df['product_id'])
img_pids  = set(image_df['product_id'])

print(f'Product IDs across datasets:')
print(f'  Text:     {len(text_pids)}')
print(f'  Image:    {len(img_pids)}')
print(f'  Metadata: {len(meta_pids)}')
print(f'  All three overlap: {len(text_pids & meta_pids & img_pids)}')
print(f'  Text-only: {len(text_pids - meta_pids - img_pids)}')
print(f'  Image-only: {len(img_pids - text_pids - meta_pids)}')
print(f'  Meta-only: {len(meta_pids - text_pids - img_pids)}')

# Synthetic overlap
synth_text_pids = set(synth_text['product_id'])
synth_meta_pids = set(synth_meta['product_id'])
print(f'\nSynthetic data:')
print(f'  Text synthetic: {len(synth_text)} rows, {len(synth_text_pids)} unique IDs')
print(f'  Meta synthetic: {len(synth_meta)} rows, {len(synth_meta_pids)} unique IDs')
print(f'  Real/synthetic ID overlap: {len(text_pids & synth_text_pids)} (should be 0)')
print(f'  Synth text/meta ID overlap: {len(synth_text_pids & synth_meta_pids)}')

# Duplicate product_id warnings
for name, df in [('synth_text', synth_text), ('synth_meta', synth_meta)]:
    dupes = df['product_id'].duplicated().sum()
    if dupes > 0:
        print(f'  WARNING: {name} has {dupes} duplicate product_ids')

## 2. Data Quality — Text Features

In [ ]:
# Text length distributions (real vs synthetic)
text_df['_text'] = (
    text_df[['title_cleaned', 'description_cleaned']].fillna('').agg(' '.join, axis=1)
)
synth_text['_text'] = (
    synth_text[['title_cleaned', 'description_cleaned']].fillna('').agg(' '.join, axis=1)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Real text lengths by class
for label, color, name in [(0, '#2ecc71', 'Legit'), (1, '#e74c3c', 'Fraud')]:
    subset = text_df[text_df['fraud_label'] == label]['_text'].str.len()
    axes[0].hist(subset, bins=50, alpha=0.6, color=color, label=f'{name} (n={len(subset)})')
axes[0].set_title('Real Data — Text Length Distribution', fontweight='bold')
axes[0].set_xlabel('Character count (title + description)')
axes[0].legend()

# Real fraud vs Synthetic fraud
real_fraud = text_df[text_df['fraud_label'] == 1]['_text'].str.len()
synth_fraud = synth_text['_text'].str.len()
axes[1].hist(real_fraud, bins=30, alpha=0.6, color='#e74c3c', label=f'Real fraud (n={len(real_fraud)})')
axes[1].hist(synth_fraud, bins=30, alpha=0.6, color='#3498db', label=f'Synthetic fraud (n={len(synth_fraud)})')
axes[1].set_title('Fraud Text Length — Real vs Synthetic', fontweight='bold')
axes[1].set_xlabel('Character count')
axes[1].legend()

plt.tight_layout()
plt.show()

# Title uniqueness
real_unique = text_df['title_cleaned'].nunique()
synth_unique = synth_text['title_cleaned'].nunique()
print(f'Title uniqueness — Real: {real_unique}/{len(text_df)} ({100*real_unique/len(text_df):.1f}%)')
print(f'Title uniqueness — Synthetic: {synth_unique}/{len(synth_text)} ({100*synth_unique/len(synth_text):.1f}%)')

# Null summary
print(f'\nNull counts in real text data:')
for col in ['title_cleaned', 'description_cleaned', 'review1_cleaned', 'review2_cleaned']:
    n_null = text_df[col].isnull().sum()
    print(f'  {col}: {n_null} ({100*n_null/len(text_df):.1f}%)')

## 3. Data Quality — Metadata Features

In [ ]:
# Key metadata feature distributions: real fraud vs legit vs synthetic
raw_features = ['listed_price', 'original_price', 'price_ratio', 'seller_rating',
                'item_rating', 'price_deviation', 'review_rating_diff']

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for i, feat in enumerate(raw_features):
    ax = axes[i]
    if feat not in meta_df.columns:
        continue

    legit_vals = meta_df[meta_df['fraud_label'] == 0][feat].dropna()
    fraud_vals = meta_df[meta_df['fraud_label'] == 1][feat].dropna()
    synth_vals = synth_meta[feat].dropna() if feat in synth_meta.columns else pd.Series()

    # Clip outliers for better visualization
    combined = pd.concat([legit_vals, fraud_vals, synth_vals])
    q01, q99 = combined.quantile(0.01), combined.quantile(0.99)

    ax.hist(legit_vals.clip(q01, q99), bins=40, alpha=0.5, color='#2ecc71', label='Real legit', density=True)
    ax.hist(fraud_vals.clip(q01, q99), bins=40, alpha=0.5, color='#e74c3c', label='Real fraud', density=True)
    if len(synth_vals) > 0:
        ax.hist(synth_vals.clip(q01, q99), bins=40, alpha=0.4, color='#3498db', label='Synthetic', density=True)
    ax.set_title(feat, fontweight='bold', fontsize=10)
    ax.legend(fontsize=7)

# Hide unused subplot
axes[-1].set_visible(False)

plt.suptitle('Metadata Feature Distributions — Real (legit/fraud) vs Synthetic', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap for metadata features
raw_features_full = ['listed_price', 'original_price', 'seller_rating', 'rating_count',
                     'item_rating', 'item_rating_count', 'review1_rating', 'review2_rating',
                     'price_deviation', 'price_ratio', 'abnormal_discount',
                     'review_rating_diff', 'seller_item_rating_gap', 'fraud_label']

corr = meta_df[raw_features_full].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=ax, square=True, linewidths=0.5,
            cbar_kws={'label': 'Correlation'})
ax.set_title('Metadata Feature Correlations (with fraud_label)', fontweight='bold')
plt.tight_layout()
plt.show()

# Top correlated features with fraud
fraud_corr = corr['fraud_label'].drop('fraud_label').abs().sort_values(ascending=False)
print('Top features correlated with fraud_label:')
for feat, val in fraud_corr.items():
    direction = '+' if corr.loc[feat, 'fraud_label'] > 0 else '-'
    print(f'  {direction} {feat}: {corr.loc[feat, "fraud_label"]:.3f}')

## 4. Post-Training Metrics

**After retraining all models**, place the prediction CSV files in the project root:
- `text_test_predictions.csv` (from text notebook)
- `image_test_predictions.csv` (from image notebook)
- `metadata_test_predictions.csv` (from metadata script)

Then re-run the cells below.

In [ ]:
# Load prediction CSVs (run AFTER training all models)
pred_files = {
    'Text':     'text_test_predictions.csv',
    'Image':    'image_test_predictions.csv',
    'Metadata': 'metadata_test_predictions.csv',
}

preds = {}
for name, fname in pred_files.items():
    p = Path(fname)
    if p.exists():
        preds[name] = pd.read_csv(p)
        print(f'{name}: loaded {len(preds[name])} predictions from {fname}')
    else:
        print(f'{name}: {fname} NOT FOUND — train the model first')

if not preds:
    print('\nNo prediction files found yet. Train models first, then re-run this cell.')

In [ ]:
# Per-model metrics summary
if preds:
    model_metrics = []
    for name, df in preds.items():
        y_true = df['fraud_label'].values
        prob_col = [c for c in df.columns if 'proba' in c][0]
        pred_col = [c for c in df.columns if 'pred' in c and 'proba' not in c][0]
        y_prob = df[prob_col].values
        y_pred = df[pred_col].values

        metrics = {
            'Model': name,
            'Samples': len(df),
            'Accuracy': accuracy_score(y_true, y_pred),
            'F1': f1_score(y_true, y_pred),
            'ROC-AUC': roc_auc_score(y_true, y_prob),
            'Avg Precision': average_precision_score(y_true, y_prob),
        }
        model_metrics.append(metrics)

    metrics_table = pd.DataFrame(model_metrics)
    print(metrics_table.to_string(index=False, float_format='{:.4f}'.format))
else:
    print('No predictions loaded — skip.')

In [ ]:
# ROC Curves — all models
if preds:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    colors = {'Text': '#e74c3c', 'Image': '#3498db', 'Metadata': '#2ecc71'}

    # Individual ROC curves
    for i, (name, df) in enumerate(preds.items()):
        y_true = df['fraud_label'].values
        prob_col = [c for c in df.columns if 'proba' in c][0]
        y_prob = df[prob_col].values

        fpr, tpr, _ = roc_curve(y_true, y_prob)
        roc_auc_val = auc(fpr, tpr)

        axes[i].plot(fpr, tpr, color=colors[name], lw=2, label=f'AUC = {roc_auc_val:.3f}')
        axes[i].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
        axes[i].set_title(f'{name} Model — ROC Curve', fontweight='bold')
        axes[i].set_xlabel('False Positive Rate')
        axes[i].set_ylabel('True Positive Rate')
        axes[i].legend(loc='lower right', fontsize=12)
        axes[i].set_xlim([-0.02, 1.02])
        axes[i].set_ylim([-0.02, 1.02])

    plt.tight_layout()
    plt.show()
else:
    print('No predictions loaded — skip.')

In [ ]:
# Precision-Recall Curves — all models
if preds:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for i, (name, df) in enumerate(preds.items()):
        y_true = df['fraud_label'].values
        prob_col = [c for c in df.columns if 'proba' in c][0]
        y_prob = df[prob_col].values

        precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
        ap = average_precision_score(y_true, y_prob)

        axes[i].plot(recall, precision, color=colors[name], lw=2, label=f'AP = {ap:.3f}')
        axes[i].axhline(y=y_true.mean(), color='gray', ls='--', lw=1, alpha=0.5, label='Baseline')
        axes[i].set_title(f'{name} Model — PR Curve', fontweight='bold')
        axes[i].set_xlabel('Recall')
        axes[i].set_ylabel('Precision')
        axes[i].legend(loc='upper right', fontsize=11)
        axes[i].set_xlim([-0.02, 1.02])
        axes[i].set_ylim([-0.02, 1.02])

    plt.tight_layout()
    plt.show()
else:
    print('No predictions loaded — skip.')

In [ ]:
# Confusion Matrices — all models
if preds:
    fig, axes = plt.subplots(1, len(preds), figsize=(6 * len(preds), 5))
    if len(preds) == 1:
        axes = [axes]

    for ax, (name, df) in zip(axes, preds.items()):
        y_true = df['fraud_label'].values
        pred_col = [c for c in df.columns if 'pred' in c and 'proba' not in c][0]
        y_pred = df[pred_col].values

        cm = confusion_matrix(y_true, y_pred)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'],
                    cbar=False, linewidths=1, linecolor='white')
        ax.set_title(f'{name} — Confusion Matrix', fontweight='bold')
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')

    plt.tight_layout()
    plt.show()
else:
    print('No predictions loaded — skip.')

## 5. Ensemble Analysis

Combines predictions from all 3 models using weighted averaging.
Only runs if all 3 prediction files are available.

In [ ]:
# Ensemble predictions
WEIGHTS = {'Text': 0.475, 'Image': 0.05, 'Metadata': 0.475}
THRESHOLD = 0.60
TEMPERATURES = {'Text': 1.0, 'Image': 1.0, 'Metadata': 1.2}

if len(preds) == 3:
    # Find common product_ids across all models
    common_pids = set(preds['Text']['product_id'])
    for name, df in preds.items():
        common_pids &= set(df['product_id'])
    print(f'Common product_ids across all 3 models: {len(common_pids)}')

    # Merge on product_id
    ensemble_df = None
    for name, df in preds.items():
        prob_col = [c for c in df.columns if 'proba' in c][0]
        sub = df[df['product_id'].isin(common_pids)][['product_id', 'fraud_label', prob_col]].copy()
        sub = sub.rename(columns={prob_col: f'{name.lower()}_prob'})
        if ensemble_df is None:
            ensemble_df = sub
        else:
            ensemble_df = ensemble_df.merge(sub[['product_id', f'{name.lower()}_prob']], on='product_id')

    # Apply temperature scaling
    import torch
    for name in ['Text', 'Image', 'Metadata']:
        col = f'{name.lower()}_prob'
        T = TEMPERATURES[name]
        if T != 1.0:
            # Convert prob to logit, scale, convert back
            p = ensemble_df[col].clip(1e-7, 1 - 1e-7)
            logit = np.log(p / (1 - p))
            ensemble_df[col] = 1 / (1 + np.exp(-logit / T))

    # Weighted average
    ensemble_df['ensemble_prob'] = sum(
        WEIGHTS[name] * ensemble_df[f'{name.lower()}_prob'] for name in WEIGHTS
    )
    ensemble_df['ensemble_pred'] = (ensemble_df['ensemble_prob'] >= THRESHOLD).astype(int)

    y_true = ensemble_df['fraud_label'].values
    y_prob = ensemble_df['ensemble_prob'].values
    y_pred = ensemble_df['ensemble_pred'].values

    print(f'\nEnsemble weights: {WEIGHTS}')
    print(f'Threshold: {THRESHOLD}')
    print(f'Temperatures: {TEMPERATURES}')
    print(f'\n{"="*50}')
    print(f'  ENSEMBLE RESULTS ({len(ensemble_df)} samples)')
    print(f'{"="*50}')
    print(f'Accuracy:  {accuracy_score(y_true, y_pred):.4f}')
    print(f'F1:        {f1_score(y_true, y_pred):.4f}')
    print(f'ROC-AUC:   {roc_auc_score(y_true, y_prob):.4f}')
    print(f'Avg Prec:  {average_precision_score(y_true, y_prob):.4f}')
    print(f'\n{classification_report(y_true, y_pred, target_names=["Legit", "Fraud"])}')
else:
    print(f'Need all 3 models — have {list(preds.keys())}. Train missing models first.')

In [ ]:
# Ensemble ROC + PR + Confusion Matrix
if len(preds) == 3:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # ROC — overlay all models + ensemble
    for name in preds:
        fpr, tpr, _ = roc_curve(y_true, ensemble_df[f'{name.lower()}_prob'].values)
        axes[0].plot(fpr, tpr, lw=1.5, alpha=0.6, label=f'{name} (AUC={auc(fpr,tpr):.3f})')
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    axes[0].plot(fpr, tpr, 'k-', lw=2.5, label=f'Ensemble (AUC={auc(fpr,tpr):.3f})')
    axes[0].plot([0,1], [0,1], 'k--', lw=1, alpha=0.3)
    axes[0].set_title('ROC — All Models + Ensemble', fontweight='bold')
    axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
    axes[0].legend(fontsize=9)

    # PR — overlay all models + ensemble
    for name in preds:
        prec, rec, _ = precision_recall_curve(y_true, ensemble_df[f'{name.lower()}_prob'].values)
        ap = average_precision_score(y_true, ensemble_df[f'{name.lower()}_prob'].values)
        axes[1].plot(rec, prec, lw=1.5, alpha=0.6, label=f'{name} (AP={ap:.3f})')
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    axes[1].plot(rec, prec, 'k-', lw=2.5, label=f'Ensemble (AP={ap:.3f})')
    axes[1].set_title('PR — All Models + Ensemble', fontweight='bold')
    axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
    axes[1].legend(fontsize=9)

    # Confusion matrix — ensemble
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2],
                xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'],
                cbar=False, linewidths=1, linecolor='white', annot_kws={'size': 14})
    axes[2].set_title(f'Ensemble Confusion Matrix (t={THRESHOLD})', fontweight='bold')
    axes[2].set_xlabel('Predicted'); axes[2].set_ylabel('Actual')

    plt.tight_layout()
    plt.show()

In [ ]:
# Threshold sweep — find optimal threshold for ensemble
if len(preds) == 3:
    thresholds = np.arange(0.30, 0.85, 0.01)
    results = []
    for t in thresholds:
        preds_t = (y_prob >= t).astype(int)
        results.append({
            'threshold': t,
            'f1': f1_score(y_true, preds_t),
            'precision': (preds_t[y_true == 1].sum() / max(preds_t.sum(), 1)),
            'recall': preds_t[y_true == 1].sum() / y_true.sum(),
            'fp': ((preds_t == 1) & (y_true == 0)).sum(),
            'fn': ((preds_t == 0) & (y_true == 1)).sum(),
        })

    sweep_df = pd.DataFrame(results)
    best_row = sweep_df.loc[sweep_df['f1'].idxmax()]

    fig, ax1 = plt.subplots(figsize=(12, 5))

    ax1.plot(sweep_df['threshold'], sweep_df['f1'], 'b-', lw=2, label='F1 Score')
    ax1.plot(sweep_df['threshold'], sweep_df['precision'], 'g--', lw=1.5, label='Precision')
    ax1.plot(sweep_df['threshold'], sweep_df['recall'], 'r--', lw=1.5, label='Recall')
    ax1.axvline(x=THRESHOLD, color='orange', ls=':', lw=2, label=f'Current ({THRESHOLD})')
    ax1.axvline(x=best_row['threshold'], color='purple', ls=':', lw=2,
                label=f'Best F1 ({best_row["threshold"]:.2f})')
    ax1.set_xlabel('Threshold', fontweight='bold')
    ax1.set_ylabel('Score', fontweight='bold')
    ax1.set_title('Ensemble Threshold Sweep', fontweight='bold')
    ax1.legend(loc='center left')
    ax1.set_xlim(0.30, 0.85)

    ax2 = ax1.twinx()
    ax2.fill_between(sweep_df['threshold'], sweep_df['fp'], alpha=0.15, color='orange', label='False Positives')
    ax2.fill_between(sweep_df['threshold'], sweep_df['fn'], alpha=0.15, color='red', label='False Negatives')
    ax2.set_ylabel('Error Count', fontweight='bold')
    ax2.legend(loc='center right')

    plt.tight_layout()
    plt.show()

    print(f'Best F1 threshold: {best_row["threshold"]:.2f} → F1={best_row["f1"]:.4f}')
    print(f'Current threshold: {THRESHOLD} → F1={sweep_df[sweep_df["threshold"].round(2) == THRESHOLD]["f1"].values[0]:.4f}')

In [ ]:
# Prediction probability distributions — per model
if len(preds) == 3:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for ax, name in zip(axes, ['Text', 'Image', 'Metadata']):
        col = f'{name.lower()}_prob'
        legit_probs = ensemble_df[ensemble_df['fraud_label'] == 0][col]
        fraud_probs = ensemble_df[ensemble_df['fraud_label'] == 1][col]

        ax.hist(legit_probs, bins=50, alpha=0.6, color='#2ecc71', label='Actually Legit', density=True)
        ax.hist(fraud_probs, bins=50, alpha=0.6, color='#e74c3c', label='Actually Fraud', density=True)
        ax.axvline(x=0.5, color='gray', ls='--', lw=1, alpha=0.5)
        ax.set_title(f'{name} — Probability Distribution', fontweight='bold')
        ax.set_xlabel('Fraud Probability')
        ax.legend(fontsize=9)

    plt.tight_layout()
    plt.show()

    # Ensemble distribution
    fig, ax = plt.subplots(figsize=(10, 4))
    legit_probs = ensemble_df[ensemble_df['fraud_label'] == 0]['ensemble_prob']
    fraud_probs = ensemble_df[ensemble_df['fraud_label'] == 1]['ensemble_prob']
    ax.hist(legit_probs, bins=50, alpha=0.6, color='#2ecc71', label='Actually Legit', density=True)
    ax.hist(fraud_probs, bins=50, alpha=0.6, color='#e74c3c', label='Actually Fraud', density=True)
    ax.axvline(x=THRESHOLD, color='orange', ls='--', lw=2, label=f'Threshold ({THRESHOLD})')
    ax.set_title('Ensemble — Probability Distribution', fontweight='bold')
    ax.set_xlabel('Ensemble Fraud Probability')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 6. Error Analysis

Inspect the most confident false positives and false negatives to understand failure modes.

In [ ]:
# Error analysis — most confident mistakes
if len(preds) == 3:
    ensemble_df_sorted = ensemble_df.copy()

    # False Positives: predicted fraud but actually legit — sorted by highest confidence
    fp_mask = (ensemble_df_sorted['ensemble_pred'] == 1) & (ensemble_df_sorted['fraud_label'] == 0)
    fps = ensemble_df_sorted[fp_mask].sort_values('ensemble_prob', ascending=False)

    # False Negatives: predicted legit but actually fraud — sorted by lowest probability
    fn_mask = (ensemble_df_sorted['ensemble_pred'] == 0) & (ensemble_df_sorted['fraud_label'] == 1)
    fns = ensemble_df_sorted[fn_mask].sort_values('ensemble_prob', ascending=True)

    print(f'False Positives: {len(fps)} (predicted fraud, actually legit)')
    print(f'False Negatives: {len(fns)} (predicted legit, actually fraud)\n')

    display_cols = ['product_id', 'text_prob', 'image_prob', 'metadata_prob', 'ensemble_prob']

    if len(fps) > 0:
        print('Top 10 False Positives (highest confidence mistakes):')
        print(fps[display_cols].head(10).to_string(index=False, float_format='{:.4f}'.format))

    if len(fns) > 0:
        print(f'\nTop 10 False Negatives (most missed fraud):')
        print(fns[display_cols].head(10).to_string(index=False, float_format='{:.4f}'.format))

    # Which model contributes most to FPs?
    if len(fps) > 0:
        print(f'\nFP model contribution (avg prob for false positives):')
        for name in ['text', 'image', 'metadata']:
            col = f'{name}_prob'
            print(f'  {name}: {fps[col].mean():.4f}')

    if len(fns) > 0:
        print(f'\nFN model contribution (avg prob for false negatives):')
        for name in ['text', 'image', 'metadata']:
            col = f'{name}_prob'
            print(f'  {name}: {fns[col].mean():.4f}')

## 7. Metrics History

Track performance across training runs by manually logging results below.
Update the dictionaries after each retraining session.

In [ ]:
# ══════════════════════════════════════════════════════════════
#  METRICS HISTORY — update after each training run
# ══════════════════════════════════════════════════════════════
# Format: {'date': 'YYYY-MM-DD', 'notes': '...', 'text_f1': X, 'image_f1': X, ...}

history = [
    # EXAMPLE (replace with actual results after training):
    # {
    #     'date': '2026-03-12',
    #     'notes': 'Baseline — k-fold CV, 300 synth, no class_weight',
    #     'text_f1': 0.0, 'text_auc': 0.0,
    #     'image_f1': 0.0, 'image_auc': 0.0,
    #     'meta_f1': 0.0, 'meta_auc': 0.0,
    #     'ensemble_f1': 0.0, 'ensemble_auc': 0.0,
    # },
]

if history:
    hist_df = pd.DataFrame(history)
    print(hist_df.to_string(index=False, float_format='{:.4f}'.format))

    # Plot F1 over time
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    x = range(len(hist_df))
    for col, label, color in [
        ('text_f1', 'Text', '#e74c3c'), ('image_f1', 'Image', '#3498db'),
        ('meta_f1', 'Metadata', '#2ecc71'), ('ensemble_f1', 'Ensemble', 'black'),
    ]:
        if col in hist_df.columns:
            axes[0].plot(x, hist_df[col], 'o-', label=label, color=color, lw=2 if 'ensemble' in col else 1.5)
    axes[0].set_title('F1 Score Over Training Runs', fontweight='bold')
    axes[0].set_xlabel('Run'); axes[0].set_ylabel('F1')
    axes[0].legend(); axes[0].set_xticks(list(x))
    axes[0].set_xticklabels(hist_df['date'], rotation=45)

    for col, label, color in [
        ('text_auc', 'Text', '#e74c3c'), ('image_auc', 'Image', '#3498db'),
        ('meta_auc', 'Metadata', '#2ecc71'), ('ensemble_auc', 'Ensemble', 'black'),
    ]:
        if col in hist_df.columns:
            axes[1].plot(x, hist_df[col], 'o-', label=label, color=color, lw=2 if 'ensemble' in col else 1.5)
    axes[1].set_title('ROC-AUC Over Training Runs', fontweight='bold')
    axes[1].set_xlabel('Run'); axes[1].set_ylabel('AUC')
    axes[1].legend(); axes[1].set_xticks(list(x))
    axes[1].set_xticklabels(hist_df['date'], rotation=45)

    plt.tight_layout()
    plt.show()
else:
    print('No history entries yet. Add results after training, then re-run.')
    print('Copy the template above and fill in your metric values.')